# Module 5: exercises

These use the messy `messy_orders.csv`. Each exercise builds on a fresh `raw` copy where needed.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

## Exercise 1

**Audit.** Load `messy_orders.csv` into `raw`. Store the number of exact duplicate rows in `n_dupes`, and the percentage
of `quantity` values that are missing (rounded to 1 decimal) in `qty_missing_pct`.

In [ ]:
import numpy as np
import pandas as pd

raw = None              # YOUR CODE HERE
n_dupes = None          # YOUR CODE HERE
qty_missing_pct = None  # YOUR CODE HERE

In [ ]:
assert n_dupes == 12
assert qty_missing_pct == 5.1, f"Got {qty_missing_pct}: percentage = share * 100"
print("✅ Correct!")

## Exercise 2

**Tidy the text.** Starting from `raw`, make `df`: drop the duplicates and reset the index, then standardise `customer`, `city` and `item`
(strip spaces, title case), and fix the `"Chiangmai"` spelling. After this, `df["city"]` should have exactly 3 distinct values.

In [ ]:
df = None   # YOUR CODE HERE

In [ ]:
assert len(df) == 300, "Remove the 12 duplicates"
assert set(df["city"]) == {"Bangkok", "Chiang Mai", "Phuket"}, set(df["city"])
assert set(df["item"]) == {"Cake", "Espresso", "Latte", "Mocha", "Tea"}, set(df["item"])
assert df["customer"].nunique() == 7
print("✅ Correct!")

## Exercise 3

**Prices and dates.** In `df`, convert `unit_price` to a numeric column (remove the `$`) and `order_date` to datetime.
Then store in `n_bad_prices` how many prices are missing after the conversion.

In [ ]:
n_bad_prices = None   # YOUR CODE HERE (after converting the columns)

In [ ]:
assert pd.api.types.is_numeric_dtype(df["unit_price"]), "unit_price should be numeric"
assert pd.api.types.is_datetime64_any_dtype(df["order_date"]), "order_date should be datetime"
assert n_bad_prices == 15
assert df["order_date"].min() == pd.Timestamp("2024-01-01")
print("✅ Correct!")

## Exercise 4

**Fill the prices.** Fill the missing `unit_price` values using each item's **median** price (look at how the lesson did it).
Afterwards `df["unit_price"]` must have no missing values, and `price_by_item` should hold the median price per item.

In [ ]:
price_by_item = None   # YOUR CODE HERE
# then fill df["unit_price"]

In [ ]:
assert df["unit_price"].isna().sum() == 0
assert price_by_item["Latte"] == 4.5 and price_by_item["Tea"] == 2.5
print("✅ Correct!")

## Exercise 5

**IQR outliers.** Write `iqr_bounds(series)` returning the tuple `(low, high)` from the rule $Q_1-1.5\,\text{IQR}$ and $Q_3+1.5\,\text{IQR}$
(ignore missing values). Use it to compute `n_outliers`: how many `quantity` values in `df` fall outside the bounds.

In [ ]:
def iqr_bounds(series):
    # YOUR CODE HERE
    pass

n_outliers = None   # YOUR CODE HERE

In [ ]:
lo, hi = iqr_bounds(pd.Series([1, 2, 3, 4, 5, 6, 7, 8, 100]))
assert lo < 1 and 8 < hi < 100, "100 should fall outside the bounds, the rest inside"
assert n_outliers == 4, f"Got {n_outliers}"
print("✅ Correct!")

## Exercise 6

**Fix quantity, then compute revenue.** Treat every `quantity` above 50 as missing, fill the missing quantities with the median
quantity, then add `revenue = quantity * unit_price`. Store the total in `total_revenue`.

In [ ]:
total_revenue = None   # YOUR CODE HERE (after fixing df)

In [ ]:
assert df["quantity"].between(1, 5).all()
assert abs(total_revenue - 3506.5) < 1e-6, f"Got {total_revenue}"
print("✅ Correct!")

## Exercise 7

**Shares and growth.** Using `df`: compute `city_share`, the percentage of total revenue earned in each city (a Series indexed by city,
summing to 100). Then compute `top_city`, the name of the city with the largest share.

In [ ]:
city_share = None   # YOUR CODE HERE
top_city = None     # YOUR CODE HERE

In [ ]:
assert abs(city_share.sum() - 100) < 1e-9
assert top_city == "Bangkok"
assert abs(city_share["Bangkok"] - 54.9) < 0.1
print("✅ Correct!")

## Exercise 8

**Merge.** Merge `df` with the `catalog` table on `item` (a left join) to make `merged`, then compute `food_revenue`:
total revenue of the `"food"` category. Check that the row count did not change.

In [ ]:
catalog = pd.DataFrame({
    "item": ["Latte", "Espresso", "Mocha", "Tea", "Cake"],
    "category": ["coffee", "coffee", "coffee", "tea", "food"],
})
merged = None         # YOUR CODE HERE
food_revenue = None   # YOUR CODE HERE

In [ ]:
assert len(merged) == len(df) == 300, "A left join with a unique key keeps the row count"
assert merged["category"].notna().all()
assert abs(food_revenue - merged[merged["item"] == "Cake"]["revenue"].sum()) < 1e-9
print("✅ Correct! Food revenue:", food_revenue)

## Exercise 9

**The whole pipeline.** Write `clean_orders(raw)` that performs *all* of the steps above and returns the final table (with a `revenue` column).
It must not modify `raw`. This function is the seed of your project's `clean.py`.

In [ ]:
def clean_orders(raw):
    # YOUR CODE HERE
    pass

In [ ]:
fresh = pd.read_csv(DATA / "messy_orders.csv")
snapshot = fresh.copy()
out = clean_orders(fresh)
assert fresh.equals(snapshot), "clean_orders must not modify its input"
assert len(out) == 300 and out.duplicated().sum() == 0
assert out[["quantity", "unit_price", "revenue"]].notna().all().all()
assert abs(out["revenue"].sum() - 3506.5) < 1e-6
print("✅ Correct! Copy this into projects/template/src/myproject/clean.py and adapt it to your data.")